In [17]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 119.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 90.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 57.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11/11 [matplotlib]1 [matplotlib]]
Note: you may need to restart the kernel to use updated packages.


In [18]:
import os

for root, dirs, files in os.walk("lvis_16k_dataset/"):
    print(root)
    for d in dirs:
        print("  📁", d)
    for f in files[:5]:
        print("  📄", f)

lvis_16k_dataset/
  📁 images
  📁 .ipynb_checkpoints
  📄 annotations.json
lvis_16k_dataset/images
  📄 000000000049.jpg
  📄 000000000192.jpg
  📄 000000000196.jpg
  📄 000000000257.jpg
  📄 000000000357.jpg
lvis_16k_dataset/.ipynb_checkpoints


In [19]:
IMAGE_DIR = "lvis_16k_dataset/images"
ANNOTATION_FILE = "lvis_16k_dataset/annotations.json"

import os

print("Images count:", len(os.listdir(IMAGE_DIR)))
print("Annotation file exists:", os.path.exists(ANNOTATION_FILE))

Images count: 15744
Annotation file exists: True


In [20]:
from pycocotools.coco import COCO

ann_file = "lvis_16k_dataset/annotations.json"
coco = COCO(ann_file)

image_ids = coco.getImgIds()

loading annotations into memory...
Done (t=4.38s)
creating index...
index created!


In [21]:
import torchvision
import torch

model = torchvision.models.detection.maskrcnn_resnet50_fpn(pretrained=True)
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

/venv/main/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/venv/main/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=MaskRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth" to /root/.cache/torch/hub/checkpoints/maskrcnn_resnet50_fpn_coco-bf2d0c1e.pth


100%|██████████| 170M/170M [00:01<00:00, 102MB/s]  


MaskRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(in

In [22]:
from PIL import Image
import torchvision.transforms as T

transform = T.Compose([T.ToTensor()])

def get_predictions(img_path):
    image = Image.open(img_path).convert("RGB")
    img_tensor = transform(image).to(device)

    with torch.no_grad():
        outputs = model([img_tensor])[0]

    return outputs

In [23]:
def format_predictions(outputs, image_id):
    preds = []

    for i in range(len(outputs["boxes"])):
        score = outputs["scores"][i].item()

        if score < 0.5:
            continue

        box = outputs["boxes"][i].tolist()
        label = outputs["labels"][i].item()

        x1, y1, x2, y2 = box

        preds.append({
            "image_id": image_id,
            "category_id": label,
            "bbox": [x1, y1, x2 - x1, y2 - y1],
            "score": score
        })

    return preds

In [24]:
import os

def get_image_path(img_info):
    file_name = os.path.basename(img_info["coco_url"])
    return f"lvis_16k_dataset/images/{file_name}"   # or change folder if needed

In [25]:
from tqdm import tqdm

all_predictions = []

for img_id in tqdm(image_ids, desc="Running Inference"):
    img_info = coco.loadImgs(img_id)[0]
    img_path = get_image_path(img_info)

    outputs = get_predictions(img_path)
    preds = format_predictions(outputs, img_id)

    all_predictions.extend(preds)

Running Inference: 100%|██████████| 15744/15744 [14:59<00:00, 17.51it/s]


In [26]:
import json

with open("rcnn_results.json", "w") as f:
    json.dump(all_predictions, f)

In [29]:
import numpy as np
import pandas as pd
from collections import defaultdict
from pycocotools.cocoeval import COCOeval

In [31]:
coco_gt = coco
coco_dt = coco_gt.loadRes("rcnn_results.json")

Loading and preparing results...
DONE (t=1.67s)
creating index...
index created!


In [33]:
# Fix missing 'iscrowd' in annotations

for ann in coco.dataset['annotations']:
    if 'iscrowd' not in ann:
        ann['iscrowd'] = 0

# Re-create COCO object to apply changes
coco = COCO()
coco.dataset = coco_gt.dataset
coco.createIndex()

creating index...
index created!


In [35]:
from tqdm import tqdm

coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')

img_ids = coco_gt.getImgIds()
cat_ids = coco_gt.getCatIds()

coco_eval.params.imgIds = img_ids
coco_eval.params.catIds = cat_ids

print("🔄 Evaluating per image...")

# Inject progress tracking
coco_eval.ious = {}

for img_id in tqdm(img_ids, desc="COCO Eval (IoU Computation)"):
    for cat_id in cat_ids:
        coco_eval.ious[(img_id, cat_id)] = coco_eval.computeIoU(img_id, cat_id)

# Continue normal flow
coco_eval.evaluate()
coco_eval.accumulate()
coco_eval.summarize()

overall_map = coco_eval.stats[0]
ap75 = coco_eval.stats[2]

🔄 Evaluating per image...


COCO Eval (IoU Computation): 100%|██████████| 15744/15744 [02:00<00:00, 130.32it/s]


Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=244.29s).
Accumulating evaluation results...
DONE (t=18.44s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=  1 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets= 10 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.001
 Average Recall     (AR) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Recall     (AR) @[ IoU=0.50:0.95 | area=medium | maxDets=

In [36]:
cat_counts = defaultdict(int)

for ann in coco.dataset['annotations']:
    cat_counts[ann['category_id']] += 1

sorted_cats = sorted(cat_counts.items(), key=lambda x: x[1])

n = len(sorted_cats)

rare = [c[0] for c in sorted_cats[:n//3]]
common = [c[0] for c in sorted_cats[n//3:2*n//3]]
frequent = [c[0] for c in sorted_cats[2*n//3:]]

In [38]:
def evaluate_subset(cat_ids):
    coco_eval = COCOeval(coco_gt, coco_dt, iouType='bbox')
    
    coco_eval.params.catIds = cat_ids
    
    coco_eval.evaluate()
    coco_eval.accumulate()
    
    # 🔥 FIX: handle empty stats
    if coco_eval.stats is None or len(coco_eval.stats) == 0:
        print(f"⚠️ No valid results for categories: {cat_ids[:5]}...")
        return 0.0
    
    return coco_eval.stats[0]

# Get categories present in predictions
pred_cat_ids = set([ann['category_id'] for ann in coco_dt.dataset['annotations']])

# Filter subsets
rare = [c for c in rare if c in pred_cat_ids]
common = [c for c in common if c in pred_cat_ids]
frequent = [c for c in frequent if c in pred_cat_ids]    

map_rare = evaluate_subset(rare)
map_common = evaluate_subset(common)
map_frequent = evaluate_subset(frequent)

Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=3.98s).
Accumulating evaluation results...
DONE (t=1.08s).
⚠️ No valid results for categories: [40, 78, 52, 49, 31]...
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=5.77s).
Accumulating evaluation results...
DONE (t=2.29s).
⚠️ No valid results for categories: [62, 67, 55, 21, 54]...
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=7.83s).
Accumulating evaluation results...
DONE (t=1.94s).
⚠️ No valid results for categories: [32, 73, 65, 5, 56]...


In [39]:
import time
from tqdm import tqdm
import os

latencies = []

# Optional: limit images to speed up (remove if you want full dataset)
sample_ids = image_ids[:200]   # you can increase later

for img_id in tqdm(sample_ids, desc="Measuring Latency"):
    img_info = coco.loadImgs(img_id)[0]
    img_path = get_image_path(img_info)

    if not os.path.exists(img_path):
        continue

    start = time.time()

    _ = get_predictions(img_path)

    end = time.time()

    latency_ms = (end - start) * 1000
    latencies.append(latency_ms)

# Final average latency
avg_latency = sum(latencies) / len(latencies)

print(f"✅ Average Latency: {avg_latency:.2f} ms")

Measuring Latency: 100%|██████████| 200/200 [00:11<00:00, 17.39it/s]

✅ Average Latency: 57.07 ms


In [40]:
results = [
    ["Overall mAP", overall_map, "IoU=0.50:0.95"],
    ["mAP_rare", map_rare, "Zero-Shot Success"],
    ["mAP_common", map_common, "Common Categories"],
    ["mAP_frequent", map_frequent, "Frequent Categories"],
    ["mIoU Proxy (AP75)", ap75, "Geometric Accuracy"],
    ["Avg Latency (ms)", avg_latency, "Inference Speed"],
    ["Baseline Type", "Mask R-CNN", "Closed-Set Comparison"]
]

df = pd.DataFrame(results, columns=["Metric", "Value", "Context"])
df

,Metric,Value,Context
0,Overall mAP,0.000007,IoU=0.50:0.95
1,mAP_rare,0.0,Zero-Shot Success
2,mAP_common,0.0,Common Categories
3,mAP_frequent,0.0,Frequent Categories
4,mIoU Proxy (AP75),0.000008,Geometric Accuracy
5,Avg Latency (ms),57.071937,Inference Speed
6,Baseline Type,Mask R-CNN,Closed-Set Comparison


In [ ]:
df.to_csv("final_rcnn_results.csv", index=False)

print("✅ Results saved to final_results.csv")